In [1]:
import os
import re
import json
import random
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
OUT_DIR = "qwen3b_subtask1_circuit"

SEED = 42
MAX_LEN = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

ICL_SHOTS = 4
YES_TEXT = " yes"
NO_TEXT = " no"

N_ANALYSIS_EXAMPLES = 12
N_MEAN_EXAMPLES = 12


# ============================================================
# UTILS
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    raise ValueError("Unsupported model architecture")


def get_final_norm(model):
    if hasattr(model, "model") and hasattr(model.model, "norm"):
        return model.model.norm
    if hasattr(model, "transformer") and hasattr(model.transformer, "ln_f"):
        return model.transformer.ln_f
    return None


def yes_no_token_ids(tokenizer):
    yes_ids = tokenizer.encode(YES_TEXT, add_special_tokens=False)
    no_ids = tokenizer.encode(NO_TEXT, add_special_tokens=False)
    if not yes_ids or not no_ids:
        raise ValueError("Could not tokenize yes/no labels")
    return yes_ids[0], no_ids[0]


def tokenize_prompt(tokenizer, prompt, device):
    toks = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )
    return {k: v.to(device) for k, v in toks.items()}


# ============================================================
# PROMPTS
# ============================================================
def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos)
    rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict formal logic reasoner. "
                "Decide only whether the conclusion logically follows from the premises. "
                "Ignore plausibility and world knowledge. "
                "Reply with only yes or no."
            ),
        }
    ]
    for ex in chosen:
        ans = "yes" if ex["validity"] else "no"
        messages.append(
            {"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."}
        )
        messages.append({"role": "assistant", "content": ans})
    messages.append(
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    )
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def split_sentences(text):
    return [x.strip() for x in re.split(r"(?<=[.!?])\s+", text.strip()) if x.strip()]


def extract_terms(sentence):
    """
    Very simple noun-phrase-ish extraction:
    collect multiword spans after cue phrases.
    This is heuristic but better than shuffling words.
    """
    patterns = [
        r"all (.+?) are (.+?)(?:\.|$)",
        r"every (.+?) is (.+?)(?:\.|$)",
        r"every single (.+?) is (.+?)(?:\.|$)",
        r"no (.+?) are (.+?)(?:\.|$)",
        r"no (.+?) is (.+?)(?:\.|$)",
        r"some (.+?) are (.+?)(?:\.|$)",
        r"some (.+?) is (.+?)(?:\.|$)",
        r"some (.+?) are not (.+?)(?:\.|$)",
        r"some (.+?) is not (.+?)(?:\.|$)",
        r"anything that is (.+?) is (.+?)(?:\.|$)",
        r"anything that is a (.+?) is a (.+?)(?:\.|$)",
        r"every single thing that is a (.+?) is a (.+?)(?:\.|$)",
        r"it is the case that a portion of (.+?) are (.+?)(?:\.|$)",
    ]
    s = sentence.strip().lower()
    for pat in patterns:
        m = re.search(pat, s)
        if m:
            return [m.group(1).strip(), m.group(2).strip()]
    return []


def safe_swap_in_sentence(sentence, term_a, term_b):
    """
    Swap two exact substrings with placeholders.
    """
    placeholder_a = "__TMP_A__"
    placeholder_b = "__TMP_B__"
    if term_a not in sentence or term_b not in sentence:
        return sentence
    sent = sentence.replace(term_a, placeholder_a)
    sent = sent.replace(term_b, placeholder_b)
    sent = sent.replace(placeholder_a, term_b)
    sent = sent.replace(placeholder_b, term_a)
    return sent


def corrupt_syllogism(syllogism):
    """
    Better corruption than word shuffle.
    Strategy:
      1. Split into sentences.
      2. If there are at least 3 sentences, try to corrupt only premise 2
         by swapping its two extracted terms.
      3. If extraction fails, swap premise 1 and premise 2.
    """
    sents = split_sentences(syllogism)
    if len(sents) < 3:
        return syllogism

    premise2 = sents[1]
    terms = extract_terms(premise2)

    if len(terms) == 2:
        a, b = terms[0], terms[1]
        corrupted_p2 = safe_swap_in_sentence(premise2, a, b)
        if corrupted_p2 != premise2:
            sents[1] = corrupted_p2
            return " ".join(sents)

    # fallback
    sents[0], sents[1] = sents[1], sents[0]
    return " ".join(sents)


def build_corrupted_prompt(tokenizer, train_data, syllogism):
    corrupted = corrupt_syllogism(syllogism)
    return build_prompt(tokenizer, train_data, corrupted), corrupted


# ============================================================
# LOGIT DIFF
# ============================================================
@torch.no_grad()
def get_logit_diff(model, tokenizer, prompt):
    inputs = tokenize_prompt(tokenizer, prompt, model.device)
    out = model(**inputs, use_cache=False)
    logits = out.logits[0, -1, :].float().cpu()
    yes_id, no_id = yes_no_token_ids(tokenizer)
    return float(logits[yes_id] - logits[no_id])


# ============================================================
# CACHING
# ============================================================
@dataclass
class ForwardCache:
    layer_out: dict
    attn_out: dict
    mlp_out: dict
    hidden_states: list


class ActivationCache:
    def __init__(self):
        self.layer_out = {}
        self.attn_out = {}
        self.mlp_out = {}
        self.handles = []

    def register(self, model):
        layers = get_layers(model)
        for i, layer in enumerate(layers):
            self.handles.append(layer.register_forward_hook(self.make_layer_hook(i)))
            if hasattr(layer, "self_attn"):
                self.handles.append(layer.self_attn.register_forward_hook(self.make_attn_hook(i)))
            if hasattr(layer, "mlp"):
                self.handles.append(layer.mlp.register_forward_hook(self.make_mlp_hook(i)))

    def make_layer_hook(self, idx):
        def hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            self.layer_out[idx] = x.detach().cpu()
        return hook

    def make_attn_hook(self, idx):
        def hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            self.attn_out[idx] = x.detach().cpu()
        return hook

    def make_mlp_hook(self, idx):
        def hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            self.mlp_out[idx] = x.detach().cpu()
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []


@torch.no_grad()
def run_with_cache(model, tokenizer, prompt):
    cache = ActivationCache()
    cache.register(model)

    inputs = tokenize_prompt(tokenizer, prompt, model.device)
    out = model(**inputs, use_cache=False, output_hidden_states=True)
    logits = out.logits[0, -1, :].float().cpu()

    yes_id, no_id = yes_no_token_ids(tokenizer)
    ld = float(logits[yes_id] - logits[no_id])

    fc = ForwardCache(
        layer_out=dict(cache.layer_out),
        attn_out=dict(cache.attn_out),
        mlp_out=dict(cache.mlp_out),
        hidden_states=[h.detach().cpu() for h in out.hidden_states],
    )
    cache.remove()
    return fc, ld


# ============================================================
# PATCHING
# ============================================================
class ReplaceOutputHook:
    def __init__(self, patch_tensor):
        self.patch_tensor = patch_tensor

    def __call__(self, module, inputs, output):
        patch = self.patch_tensor
        if isinstance(output, tuple):
            return (patch.to(output[0].device),) + output[1:]
        return patch.to(output.device)


@torch.no_grad()
def patch_and_score(model, tokenizer, corrupted_prompt, patch_tensor, module):
    hook = ReplaceOutputHook(patch_tensor)
    handle = module.register_forward_hook(hook)

    inputs = tokenize_prompt(tokenizer, corrupted_prompt, model.device)
    out = model(**inputs, use_cache=False)
    handle.remove()

    logits = out.logits[0, -1, :].float().cpu()
    yes_id, no_id = yes_no_token_ids(tokenizer)
    return float(logits[yes_id] - logits[no_id])


def run_activation_patching(model, tokenizer, train_data, examples, n_layers):
    layers = get_layers(model)
    results = {
        "layer": np.zeros(n_layers),
        "attn": np.zeros(n_layers),
        "mlp": np.zeros(n_layers),
    }
    example_logs = []
    count = 0

    for i, ex in enumerate(examples):
        clean_prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        corr_prompt, corrupted_syllogism = build_corrupted_prompt(tokenizer, train_data, ex["syllogism"])

        clean_cache, clean_ld = run_with_cache(model, tokenizer, clean_prompt)
        _, corr_ld = run_with_cache(model, tokenizer, corr_prompt)

        baseline_gap = clean_ld - corr_ld
        example_logs.append({
            "id": ex["id"],
            "validity": ex["validity"],
            "plausibility": ex["plausibility"],
            "original_syllogism": ex["syllogism"],
            "corrupted_syllogism": corrupted_syllogism,
            "clean_logit_diff": clean_ld,
            "corrupted_logit_diff": corr_ld,
            "baseline_gap": baseline_gap,
        })

        if abs(baseline_gap) < 1e-6:
            continue

        for l in range(n_layers):
            if l in clean_cache.layer_out:
                patched_ld = patch_and_score(
                    model, tokenizer, corr_prompt, clean_cache.layer_out[l], layers[l]
                )
                results["layer"][l] += (patched_ld - corr_ld) / baseline_gap

            if hasattr(layers[l], "self_attn") and l in clean_cache.attn_out:
                patched_ld = patch_and_score(
                    model, tokenizer, corr_prompt, clean_cache.attn_out[l], layers[l].self_attn
                )
                results["attn"][l] += (patched_ld - corr_ld) / baseline_gap

            if hasattr(layers[l], "mlp") and l in clean_cache.mlp_out:
                patched_ld = patch_and_score(
                    model, tokenizer, corr_prompt, clean_cache.mlp_out[l], layers[l].mlp
                )
                results["mlp"][l] += (patched_ld - corr_ld) / baseline_gap

        count += 1
        if (i + 1) % 2 == 0:
            print(f"  patching {i+1}/{len(examples)}")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if count > 0:
        for k in results:
            results[k] /= count

    return results, example_logs


# ============================================================
# LOGIT LENS / TRAJECTORY
# ============================================================
@torch.no_grad()
def logit_lens_summary(model, tokenizer, prompt, top_k=5):
    inputs = tokenize_prompt(tokenizer, prompt, model.device)
    out = model(**inputs, use_cache=False, output_hidden_states=True)

    hs = out.hidden_states
    unembed = model.lm_head.weight.float().cpu()
    final_norm = get_final_norm(model)
    yes_id, no_id = yes_no_token_ids(tokenizer)

    traj = {}
    for l, h in enumerate(hs):
        vec = h[0, -1, :].float()
        if final_norm is not None:
            vec = final_norm(vec.unsqueeze(0).to(model.device)).squeeze(0).float().cpu()
        logits = torch.matmul(unembed, vec)

        topv, topi = torch.topk(logits, top_k)
        preds = [(tokenizer.decode([int(i)]), float(v)) for i, v in zip(topi.tolist(), topv.tolist())]

        traj[l] = {
            "yes_logit": float(logits[yes_id]),
            "no_logit": float(logits[no_id]),
            "diff": float(logits[yes_id] - logits[no_id]),
            "top_tokens": preds,
        }
    return traj


# ============================================================
# MEAN ABLATION
# ============================================================
class MeanAblationHook:
    def __init__(self, mean_tensor):
        self.mean_tensor = mean_tensor

    def __call__(self, module, inputs, output):
        target = self.mean_tensor
        if isinstance(output, tuple):
            patched = target.to(output[0].device).expand_as(output[0])
            return (patched,) + output[1:]
        return target.to(output.device).expand_as(output)


@torch.no_grad()
def compute_mean_outputs(model, tokenizer, train_data, examples, n_layers):
    attn_store = defaultdict(list)
    mlp_store = defaultdict(list)
    layers = get_layers(model)

    for ex in examples:
        prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        cache, _ = run_with_cache(model, tokenizer, prompt)

        for l in range(n_layers):
            if l in cache.attn_out:
                attn_store[l].append(cache.attn_out[l])
            if l in cache.mlp_out:
                mlp_store[l].append(cache.mlp_out[l])

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_attn = {}
    mean_mlp = {}
    for l in range(n_layers):
        if attn_store[l]:
            mean_attn[l] = torch.stack(attn_store[l]).mean(dim=0)
        if mlp_store[l]:
            mean_mlp[l] = torch.stack(mlp_store[l]).mean(dim=0)

    return mean_attn, mean_mlp


@torch.no_grad()
def test_circuit_sufficiency(model, tokenizer, train_data, examples, top_components, mean_attn, mean_mlp, n_layers):
    layers = get_layers(model)
    clean_sum = 0.0
    ablated_sum = 0.0
    count = 0

    non_top = []
    for l in range(n_layers):
        if (l, "attn") not in top_components:
            non_top.append((l, "attn"))
        if (l, "mlp") not in top_components:
            non_top.append((l, "mlp"))

    for ex in examples:
        prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        clean_ld = get_logit_diff(model, tokenizer, prompt)

        handles = []
        for l, typ in non_top:
            if typ == "attn" and l in mean_attn:
                handles.append(layers[l].self_attn.register_forward_hook(MeanAblationHook(mean_attn[l])))
            elif typ == "mlp" and l in mean_mlp:
                handles.append(layers[l].mlp.register_forward_hook(MeanAblationHook(mean_mlp[l])))

        ablated_ld = get_logit_diff(model, tokenizer, prompt)

        for h in handles:
            h.remove()

        clean_sum += clean_ld
        ablated_sum += ablated_ld
        count += 1

    avg_clean = clean_sum / max(count, 1)
    avg_ablated = ablated_sum / max(count, 1)
    recovery = avg_ablated / avg_clean if abs(avg_clean) > 1e-6 else 0.0

    return {
        "avg_clean_ld": avg_clean,
        "avg_top_only_ld": avg_ablated,
        "recovery_fraction": recovery,
        "n_examples": count,
    }


@torch.no_grad()
def test_circuit_necessity(model, tokenizer, train_data, examples, top_components, mean_attn, mean_mlp):
    layers = get_layers(model)
    clean_sum = 0.0
    ablated_sum = 0.0
    count = 0

    for ex in examples:
        prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        clean_ld = get_logit_diff(model, tokenizer, prompt)

        handles = []
        for l, typ in top_components:
            if typ == "attn" and l in mean_attn:
                handles.append(layers[l].self_attn.register_forward_hook(MeanAblationHook(mean_attn[l])))
            elif typ == "mlp" and l in mean_mlp:
                handles.append(layers[l].mlp.register_forward_hook(MeanAblationHook(mean_mlp[l])))

        ablated_ld = get_logit_diff(model, tokenizer, prompt)

        for h in handles:
            h.remove()

        clean_sum += clean_ld
        ablated_sum += ablated_ld
        count += 1

    avg_clean = clean_sum / max(count, 1)
    avg_ablated = ablated_sum / max(count, 1)
    destruction = 1.0 - (avg_ablated / avg_clean) if abs(avg_clean) > 1e-6 else 0.0

    return {
        "avg_clean_ld": avg_clean,
        "avg_ablated_ld": avg_ablated,
        "destruction_fraction": destruction,
        "n_examples": count,
    }


# ============================================================
# DATA SAMPLING
# ============================================================
def stratified_examples(test_data, n_total, seed):
    rng = random.Random(seed)
    groups = {
        "valid_plausible": [x for x in test_data if x["validity"] and x["plausibility"]],
        "valid_implausible": [x for x in test_data if x["validity"] and not x["plausibility"]],
        "invalid_plausible": [x for x in test_data if (not x["validity"]) and x["plausibility"]],
        "invalid_implausible": [x for x in test_data if (not x["validity"]) and (not x["plausibility"])],
    }
    per_group = max(1, n_total // 4)
    picked = []
    for _, items in groups.items():
        rng.shuffle(items)
        picked.extend(items[:per_group])
    rng.shuffle(picked)
    return picked


# ============================================================
# MAIN
# ============================================================
def main():
    set_seed(SEED)
    os.makedirs(OUT_DIR, exist_ok=True)

    print(f"Device: {DEVICE}")
    print(f"Loading model: {MODEL_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    yes_id, no_id = yes_no_token_ids(tokenizer)
    print(f"yes token id: {yes_id} -> {tokenizer.decode([yes_id])!r}")
    print(f" no token id: {no_id} -> {tokenizer.decode([no_id])!r}")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    model.eval()
    model.config.use_cache = False

    train_data = load_json(TRAIN_JSON)
    test_data = load_json(TEST_JSON)
    n_layers = len(get_layers(model))

    print(f"Train: {len(train_data)} | Test: {len(test_data)} | Layers: {n_layers}")

    analysis_examples = stratified_examples(test_data, N_ANALYSIS_EXAMPLES, SEED)
    mean_examples = stratified_examples(test_data, N_MEAN_EXAMPLES, SEED + 1)

    # --------------------------------------------------------
    # 1. activation patching
    # --------------------------------------------------------
    print("\n=== ACTIVATION PATCHING ===")
    patch_results, example_logs = run_activation_patching(
        model, tokenizer, train_data, analysis_examples, n_layers
    )

    print(f"\n{'Layer':>5} {'LayerOut':>10} {'Attn':>10} {'MLP':>10}")
    for l in range(n_layers):
        print(
            f"{l:>5} "
            f"{patch_results['layer'][l]:>10.4f} "
            f"{patch_results['attn'][l]:>10.4f} "
            f"{patch_results['mlp'][l]:>10.4f}"
        )

    save_json(
        {
            "layer_out_recovery": patch_results["layer"].tolist(),
            "attn_recovery": patch_results["attn"].tolist(),
            "mlp_recovery": patch_results["mlp"].tolist(),
            "example_logs": example_logs,
        },
        os.path.join(OUT_DIR, "activation_patching.json"),
    )

    # --------------------------------------------------------
    # 2. top components
    # --------------------------------------------------------
    all_components = []
    for l in range(n_layers):
        all_components.append((l, "attn", abs(float(patch_results["attn"][l]))))
        all_components.append((l, "mlp", abs(float(patch_results["mlp"][l]))))
    all_components.sort(key=lambda x: x[2], reverse=True)
    top_components_list = all_components[:8]
    top_components = {(l, typ) for l, typ, _ in top_components_list}

    print("\nTop components:")
    for l, typ, score in top_components_list:
        print(f"  Layer {l:>2} {typ:>4}  score={score:.4f}")

    # --------------------------------------------------------
    # 3. mean ablation
    # --------------------------------------------------------
    print("\n=== COMPUTING MEAN OUTPUTS ===")
    mean_attn, mean_mlp = compute_mean_outputs(
        model, tokenizer, train_data, mean_examples, n_layers
    )

    print("\n=== SUFFICIENCY ===")
    suff = test_circuit_sufficiency(
        model, tokenizer, train_data, analysis_examples[:8],
        top_components, mean_attn, mean_mlp, n_layers
    )
    print(suff)

    print("\n=== NECESSITY ===")
    nec = test_circuit_necessity(
        model, tokenizer, train_data, analysis_examples[:8],
        top_components, mean_attn, mean_mlp
    )
    print(nec)

    save_json(
        {
            "top_components": [
                {"layer": l, "type": typ, "score": score}
                for l, typ, score in top_components_list
            ],
            "sufficiency": suff,
            "necessity": nec,
        },
        os.path.join(OUT_DIR, "top_component_ablation.json"),
    )

    # --------------------------------------------------------
    # 4. logit lens / trajectory
    # --------------------------------------------------------
    print("\n=== LOGIT LENS TRAJECTORIES ===")
    traj_results = []
    for ex in analysis_examples[:6]:
        prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        traj = logit_lens_summary(model, tokenizer, prompt, top_k=5)
        traj_results.append(
            {
                "id": ex["id"],
                "validity": ex["validity"],
                "plausibility": ex["plausibility"],
                "trajectory": {str(k): v for k, v in traj.items()},
            }
        )
        print(f"\nExample {ex['id'][:8]}...")
        step = max(1, len(traj) // 8)
        for l in sorted(traj.keys())[::step]:
            row = traj[l]
            print(
                f"  L{l:>2} | yes={row['yes_logit']:.3f} "
                f"no={row['no_logit']:.3f} diff={row['diff']:.3f} "
                f"top1={row['top_tokens'][0][0]!r}"
            )

    save_json(traj_results, os.path.join(OUT_DIR, "logit_lens.json"))

    print(f"\nDone. Results saved in: {OUT_DIR}")


if __name__ == "__main__":
    main()

Device: cuda
Loading model: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Train: 960 | Test: 191 | Layers: 36

=== ACTIVATION PATCHING ===
  patching 2/12
  patching 4/12
  patching 6/12
  patching 8/12
  patching 10/12
  patching 12/12

Layer   LayerOut       Attn        MLP
    0     1.0000     1.0940     1.3068
    1     1.0000    -0.2453    -0.0441
    2     1.0000    -0.2413     0.0908
    3     1.0000    -0.1494     0.1745
    4     1.0000     0.2438     0.0218
    5     1.0000     0.1998     0.0858
    6     1.0000     0.2491     0.0781
    7     1.0000     0.2682     0.3906
    8     1.0000     0.1232     0.3132
    9     1.0000     0.0483    -0.0325
   10     1.0000     0.0980    -0.2510
   11     1.0000     0.2381    -0.2895
   12     1.0000    -0.0838     0.3666
   13     1.0000    -0.0463     0.5556
   14     1.0000     0.2828     0.4654
   15     1.0000     0.4471    -0.2316
   16     1.0000    -0.3126    -0.6844
   17     1.0000     0.3444     0.3885
   18     1.0000     0.7300     0.0143
   19     1.0000     0.0600     0.5099
   20     1.0000 

RuntimeError: stack expects each tensor to be equal size, but got [1, 261, 2048] at entry 0 and [1, 262, 2048] at entry 1

In [2]:
import os
import re
import json
import random
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
OUT_DIR = "qwen3b_subtask1_circuit"

SEED = 42
MAX_LEN = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

ICL_SHOTS = 4
YES_TEXT = " yes"
NO_TEXT = " no"

N_VALIDITY_EXAMPLES = 12   # for clean->corrupt (logic) patching
N_CONTENT_PAIRS = 12       # for content-effect patching
N_MEAN_EXAMPLES = 12
N_HEAD_TOP_LAYERS = 5      # how many top layers to do per-head analysis on

# Optional: path to a trained LoRA adapter for the comparison step
LORA_ADAPTER_PATH = None   # e.g. "lora_contrastive_output/final"


# ============================================================
# UTILS
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    # PEFT-wrapped
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        inner = model.base_model.model
        if hasattr(inner, "model") and hasattr(inner.model, "layers"):
            return inner.model.layers
    raise ValueError("Unsupported model architecture")


def get_final_norm(model):
    if hasattr(model, "model") and hasattr(model.model, "norm"):
        return model.model.norm
    if hasattr(model, "base_model"):
        inner = model.base_model.model
        if hasattr(inner, "model") and hasattr(inner.model, "norm"):
            return inner.model.norm
    return None


def yes_no_token_ids(tokenizer):
    yes_ids = tokenizer.encode(YES_TEXT, add_special_tokens=False)
    no_ids = tokenizer.encode(NO_TEXT, add_special_tokens=False)
    return yes_ids[0], no_ids[0]


def tokenize_prompt(tokenizer, prompt, device):
    toks = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False,
        truncation=True, max_length=MAX_LEN,
    )
    return {k: v.to(device) for k, v in toks.items()}


# ============================================================
# PROMPTS
# ============================================================
def build_icl_examples(train_data, shots, seed):
    rng = random.Random(seed)
    pos = [x for x in train_data if x["validity"] is True]
    neg = [x for x in train_data if x["validity"] is False]
    rng.shuffle(pos); rng.shuffle(neg)
    half = shots // 2
    chosen = pos[:half] + neg[:shots - half]
    rng.shuffle(chosen)
    return chosen


def build_prompt(tokenizer, train_data, syllogism):
    chosen = build_icl_examples(train_data, ICL_SHOTS, SEED)
    messages = [{
        "role": "system",
        "content": (
            "You are a strict formal logic reasoner. "
            "Decide only whether the conclusion logically follows from the premises. "
            "Ignore plausibility and world knowledge. Reply with only yes or no."
        ),
    }]
    for ex in chosen:
        ans = "yes" if ex["validity"] else "no"
        messages.append({"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."})
        messages.append({"role": "assistant", "content": ans})
    messages.append({"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def split_sentences(text):
    return [x.strip() for x in re.split(r"(?<=[.!?])\s+", text.strip()) if x.strip()]


# ============================================================
# CORRUPTION: quantifier flip (changes validity, keeps content)
# ============================================================
QUANTIFIER_FLIPS = [
    (r"\ball (.+?) are\b", r"some \1 are not"),
    (r"\bevery (.+?) is\b", r"some \1 is not"),
    (r"\bno (.+?) are\b", r"some \1 are"),
    (r"\bno (.+?) is\b", r"some \1 is"),
    (r"\bevery single (.+?) is\b", r"not every \1 is"),
    (r"\banything that is (.+?) is\b", r"not everything that is \1 is"),
]

_corruption_stats = {"success": 0, "fail": 0}


def corrupt_syllogism(syllogism):
    """Flip a quantifier in premise 1 (or premise 2 as fallback)."""
    sents = split_sentences(syllogism)
    if len(sents) < 2:
        _corruption_stats["fail"] += 1
        return syllogism

    for idx in [0, 1]:  # try premise 1, then premise 2
        for pat, repl in QUANTIFIER_FLIPS:
            new_s, n = re.subn(pat, repl, sents[idx], count=1, flags=re.IGNORECASE)
            if n > 0:
                sents[idx] = new_s
                _corruption_stats["success"] += 1
                return " ".join(sents)

    _corruption_stats["fail"] += 1
    return syllogism


def build_corrupted_prompt(tokenizer, train_data, syllogism):
    corrupted = corrupt_syllogism(syllogism)
    if corrupted == syllogism:
        return None, None
    return build_prompt(tokenizer, train_data, corrupted), corrupted


# ============================================================
# LOGIT DIFF
# ============================================================
@torch.no_grad()
def get_logit_diff(model, tokenizer, prompt):
    inputs = tokenize_prompt(tokenizer, prompt, model.device)
    out = model(**inputs, use_cache=False)
    logits = out.logits[0, -1, :].float().cpu()
    yes_id, no_id = yes_no_token_ids(tokenizer)
    return float(logits[yes_id] - logits[no_id])


# ============================================================
# ACTIVATION CACHING (last-token only, memory-light)
# ============================================================
@dataclass
class ForwardCache:
    attn_last: dict   # layer -> [hidden]
    mlp_last: dict    # layer -> [hidden]
    oproj_in_last: dict  # layer -> [hidden] (input to o_proj, for per-head)


class ActivationCache:
    def __init__(self):
        self.attn_last = {}
        self.mlp_last = {}
        self.oproj_in_last = {}
        self.handles = []

    def register(self, model):
        layers = get_layers(model)
        for i, layer in enumerate(layers):
            if hasattr(layer, "self_attn"):
                self.handles.append(
                    layer.self_attn.register_forward_hook(self._attn_hook(i))
                )
                if hasattr(layer.self_attn, "o_proj"):
                    self.handles.append(
                        layer.self_attn.o_proj.register_forward_pre_hook(self._oproj_hook(i))
                    )
            if hasattr(layer, "mlp"):
                self.handles.append(
                    layer.mlp.register_forward_hook(self._mlp_hook(i))
                )

    def _attn_hook(self, idx):
        def hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            self.attn_last[idx] = x[0, -1, :].detach().cpu()
        return hook

    def _mlp_hook(self, idx):
        def hook(module, inputs, output):
            x = output[0] if isinstance(output, tuple) else output
            self.mlp_last[idx] = x[0, -1, :].detach().cpu()
        return hook

    def _oproj_hook(self, idx):
        def hook(module, inputs):
            x = inputs[0]  # [batch, seq, hidden]
            self.oproj_in_last[idx] = x[0, -1, :].detach().cpu()
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []


@torch.no_grad()
def run_with_cache(model, tokenizer, prompt):
    cache = ActivationCache()
    cache.register(model)
    inputs = tokenize_prompt(tokenizer, prompt, model.device)
    out = model(**inputs, use_cache=False)
    logits = out.logits[0, -1, :].float().cpu()
    yes_id, no_id = yes_no_token_ids(tokenizer)
    ld = float(logits[yes_id] - logits[no_id])
    fc = ForwardCache(
        attn_last=dict(cache.attn_last),
        mlp_last=dict(cache.mlp_last),
        oproj_in_last=dict(cache.oproj_in_last),
    )
    cache.remove()
    return fc, ld


# ============================================================
# PATCHING HOOKS (last-token)
# ============================================================
class LastTokenPatchHook:
    def __init__(self, patch_vec):
        self.patch_vec = patch_vec  # [hidden]

    def __call__(self, module, inputs, output):
        is_tuple = isinstance(output, tuple)
        x = output[0] if is_tuple else output
        y = x.clone()
        y[:, -1, :] = self.patch_vec.to(x.device, dtype=x.dtype)
        if is_tuple:
            return (y,) + output[1:]
        return y


class LastTokenHeadPatchHook:
    """Hook on o_proj's INPUT to patch a single head's last-token slice."""
    def __init__(self, patch_vec, head_idx, head_dim):
        self.patch_vec = patch_vec  # [head_dim]
        self.head_idx = head_idx
        self.head_dim = head_dim

    def __call__(self, module, inputs):
        x = inputs[0]  # [batch, seq, hidden]
        y = x.clone()
        s = self.head_idx * self.head_dim
        e = s + self.head_dim
        y[:, -1, s:e] = self.patch_vec.to(x.device, dtype=x.dtype)
        return (y,)


@torch.no_grad()
def patch_and_score(model, tokenizer, prompt, patch_vec, module, hook_cls=LastTokenPatchHook, **kwargs):
    hook = hook_cls(patch_vec, **kwargs) if kwargs else hook_cls(patch_vec)
    if hook_cls is LastTokenHeadPatchHook:
        handle = module.register_forward_pre_hook(hook)
    else:
        handle = module.register_forward_hook(hook)
    inputs = tokenize_prompt(tokenizer, prompt, model.device)
    out = model(**inputs, use_cache=False)
    handle.remove()
    logits = out.logits[0, -1, :].float().cpu()
    yes_id, no_id = yes_no_token_ids(tokenizer)
    return float(logits[yes_id] - logits[no_id])


# ============================================================
# VALIDITY PATCHING (clean -> corrupted, quantifier flipped)
# ============================================================
def run_validity_patching(model, tokenizer, train_data, examples, n_layers):
    layers = get_layers(model)
    results = {"attn": np.zeros(n_layers), "mlp": np.zeros(n_layers)}
    logs = []
    count = 0

    for i, ex in enumerate(examples):
        clean_prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        corr_prompt, corrupted_syl = build_corrupted_prompt(tokenizer, train_data, ex["syllogism"])
        if corr_prompt is None:
            continue

        clean_cache, clean_ld = run_with_cache(model, tokenizer, clean_prompt)
        _, corr_ld = run_with_cache(model, tokenizer, corr_prompt)

        gap = clean_ld - corr_ld
        logs.append({
            "id": ex["id"], "validity": ex["validity"], "plausibility": ex["plausibility"],
            "corrupted": corrupted_syl, "clean_ld": clean_ld, "corr_ld": corr_ld, "gap": gap,
        })
        if abs(gap) < 1e-6:
            continue

        for l in range(n_layers):
            if l in clean_cache.attn_last and hasattr(layers[l], "self_attn"):
                p = patch_and_score(model, tokenizer, corr_prompt,
                                    clean_cache.attn_last[l], layers[l].self_attn)
                results["attn"][l] += (p - corr_ld) / gap
            if l in clean_cache.mlp_last and hasattr(layers[l], "mlp"):
                p = patch_and_score(model, tokenizer, corr_prompt,
                                    clean_cache.mlp_last[l], layers[l].mlp)
                results["mlp"][l] += (p - corr_ld) / gap

        count += 1
        if (i + 1) % 2 == 0:
            print(f"  validity patching {i+1}/{len(examples)}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if count > 0:
        for k in results:
            results[k] /= count
    return results, logs, count


# ============================================================
# CONTENT-EFFECT PATCHING (matched validity, different plausibility)
# ============================================================
def make_content_pairs(test_data, n_pairs, seed):
    rng = random.Random(seed)
    vp = [x for x in test_data if x["validity"] and x["plausibility"]]
    vi = [x for x in test_data if x["validity"] and not x["plausibility"]]
    ip = [x for x in test_data if not x["validity"] and x["plausibility"]]
    ii = [x for x in test_data if not x["validity"] and not x["plausibility"]]
    rng.shuffle(vp); rng.shuffle(vi); rng.shuffle(ip); rng.shuffle(ii)
    pairs = []
    for a, b in zip(vp, vi):
        pairs.append(("valid", a, b))
    for a, b in zip(ip, ii):
        pairs.append(("invalid", a, b))
    rng.shuffle(pairs)
    return pairs[:n_pairs]


def run_content_patching(model, tokenizer, train_data, pairs, n_layers):
    layers = get_layers(model)
    results = {"attn": np.zeros(n_layers), "mlp": np.zeros(n_layers)}
    count = 0

    for i, (val_label, src_ex, tgt_ex) in enumerate(pairs):
        src_prompt = build_prompt(tokenizer, train_data, src_ex["syllogism"])
        tgt_prompt = build_prompt(tokenizer, train_data, tgt_ex["syllogism"])

        src_cache, src_ld = run_with_cache(model, tokenizer, src_prompt)
        _, tgt_ld = run_with_cache(model, tokenizer, tgt_prompt)

        gap = src_ld - tgt_ld
        if abs(gap) < 1e-6:
            continue

        for l in range(n_layers):
            if l in src_cache.attn_last and hasattr(layers[l], "self_attn"):
                p = patch_and_score(model, tokenizer, tgt_prompt,
                                    src_cache.attn_last[l], layers[l].self_attn)
                results["attn"][l] += (p - tgt_ld) / gap
            if l in src_cache.mlp_last and hasattr(layers[l], "mlp"):
                p = patch_and_score(model, tokenizer, tgt_prompt,
                                    src_cache.mlp_last[l], layers[l].mlp)
                results["mlp"][l] += (p - tgt_ld) / gap

        count += 1
        if (i + 1) % 2 == 0:
            print(f"  content patching {i+1}/{len(pairs)}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if count > 0:
        for k in results:
            results[k] /= count
    return results, count


# ============================================================
# HEAD-LEVEL PATCHING (content-effect, on top layers only)
# ============================================================
def run_head_level_content_patching(model, tokenizer, train_data, pairs, top_layer_ids):
    """For each top layer, patch each head's last-token slice individually."""
    layers = get_layers(model)
    cfg = model.config
    n_heads = getattr(cfg, "num_attention_heads", None)
    hidden = getattr(cfg, "hidden_size", None)
    if n_heads is None or hidden is None:
        raise ValueError("Can't determine head dims from config")
    head_dim = hidden // n_heads

    results = {l: np.zeros(n_heads) for l in top_layer_ids}
    count = 0

    for i, (val_label, src_ex, tgt_ex) in enumerate(pairs):
        src_prompt = build_prompt(tokenizer, train_data, src_ex["syllogism"])
        tgt_prompt = build_prompt(tokenizer, train_data, tgt_ex["syllogism"])

        src_cache, src_ld = run_with_cache(model, tokenizer, src_prompt)
        _, tgt_ld = run_with_cache(model, tokenizer, tgt_prompt)
        gap = src_ld - tgt_ld
        if abs(gap) < 1e-6:
            continue

        for l in top_layer_ids:
            if l not in src_cache.oproj_in_last:
                continue
            src_vec = src_cache.oproj_in_last[l]  # [hidden]
            o_proj = layers[l].self_attn.o_proj
            for h in range(n_heads):
                head_vec = src_vec[h * head_dim:(h + 1) * head_dim]
                p = patch_and_score(
                    model, tokenizer, tgt_prompt, head_vec, o_proj,
                    hook_cls=LastTokenHeadPatchHook, head_idx=h, head_dim=head_dim,
                )
                results[l][h] += (p - tgt_ld) / gap

        count += 1
        if (i + 1) % 2 == 0:
            print(f"  head patching {i+1}/{len(pairs)}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if count > 0:
        for l in results:
            results[l] /= count
    return results, count, n_heads, head_dim


# ============================================================
# MEAN ABLATION (fixed: last-token only)
# ============================================================
@torch.no_grad()
def compute_mean_outputs(model, tokenizer, train_data, examples, n_layers):
    attn_store = defaultdict(list)
    mlp_store = defaultdict(list)

    for ex in examples:
        prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        cache, _ = run_with_cache(model, tokenizer, prompt)
        for l in range(n_layers):
            if l in cache.attn_last:
                attn_store[l].append(cache.attn_last[l])
            if l in cache.mlp_last:
                mlp_store[l].append(cache.mlp_last[l])
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_attn, mean_mlp = {}, {}
    for l in range(n_layers):
        if attn_store[l]:
            mean_attn[l] = torch.stack(attn_store[l]).mean(dim=0)
        if mlp_store[l]:
            mean_mlp[l] = torch.stack(mlp_store[l]).mean(dim=0)
    return mean_attn, mean_mlp


@torch.no_grad()
def test_circuit_sufficiency(model, tokenizer, train_data, examples, top_components, mean_attn, mean_mlp, n_layers):
    layers = get_layers(model)
    non_top = []
    for l in range(n_layers):
        if (l, "attn") not in top_components:
            non_top.append((l, "attn"))
        if (l, "mlp") not in top_components:
            non_top.append((l, "mlp"))

    clean_sum, ablated_sum, count = 0.0, 0.0, 0
    for ex in examples:
        prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        clean_ld = get_logit_diff(model, tokenizer, prompt)

        handles = []
        for l, typ in non_top:
            if typ == "attn" and l in mean_attn:
                handles.append(layers[l].self_attn.register_forward_hook(LastTokenPatchHook(mean_attn[l])))
            elif typ == "mlp" and l in mean_mlp:
                handles.append(layers[l].mlp.register_forward_hook(LastTokenPatchHook(mean_mlp[l])))
        ablated_ld = get_logit_diff(model, tokenizer, prompt)
        for h in handles:
            h.remove()

        clean_sum += clean_ld
        ablated_sum += ablated_ld
        count += 1

    avg_clean = clean_sum / max(count, 1)
    avg_abl = ablated_sum / max(count, 1)
    recovery = avg_abl / avg_clean if abs(avg_clean) > 1e-6 else 0.0
    return {"avg_clean_ld": avg_clean, "avg_top_only_ld": avg_abl,
            "recovery_fraction": recovery, "n_examples": count}


@torch.no_grad()
def test_circuit_necessity(model, tokenizer, train_data, examples, top_components, mean_attn, mean_mlp):
    layers = get_layers(model)
    clean_sum, ablated_sum, count = 0.0, 0.0, 0
    for ex in examples:
        prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        clean_ld = get_logit_diff(model, tokenizer, prompt)

        handles = []
        for l, typ in top_components:
            if typ == "attn" and l in mean_attn:
                handles.append(layers[l].self_attn.register_forward_hook(LastTokenPatchHook(mean_attn[l])))
            elif typ == "mlp" and l in mean_mlp:
                handles.append(layers[l].mlp.register_forward_hook(LastTokenPatchHook(mean_mlp[l])))
        ablated_ld = get_logit_diff(model, tokenizer, prompt)
        for h in handles:
            h.remove()

        clean_sum += clean_ld
        ablated_sum += ablated_ld
        count += 1

    avg_clean = clean_sum / max(count, 1)
    avg_abl = ablated_sum / max(count, 1)
    destruction = 1.0 - (avg_abl / avg_clean) if abs(avg_clean) > 1e-6 else 0.0
    return {"avg_clean_ld": avg_clean, "avg_ablated_ld": avg_abl,
            "destruction_fraction": destruction, "n_examples": count}


# ============================================================
# LOGIT LENS
# ============================================================
@torch.no_grad()
def logit_lens_summary(model, tokenizer, prompt, top_k=5):
    inputs = tokenize_prompt(tokenizer, prompt, model.device)
    out = model(**inputs, use_cache=False, output_hidden_states=True)
    hs = out.hidden_states
    unembed = model.lm_head.weight.float().cpu()
    final_norm = get_final_norm(model)
    yes_id, no_id = yes_no_token_ids(tokenizer)

    traj = {}
    for l, h in enumerate(hs):
        vec = h[0, -1, :].float()
        if final_norm is not None:
            vec = final_norm(vec.unsqueeze(0).to(model.device)).squeeze(0).float().cpu()
        logits = torch.matmul(unembed, vec)
        topv, topi = torch.topk(logits, top_k)
        preds = [(tokenizer.decode([int(i)]), float(v)) for i, v in zip(topi.tolist(), topv.tolist())]
        traj[l] = {
            "yes_logit": float(logits[yes_id]),
            "no_logit": float(logits[no_id]),
            "diff": float(logits[yes_id] - logits[no_id]),
            "top_tokens": preds,
        }
    return traj


# ============================================================
# SAMPLING
# ============================================================
def stratified_examples(test_data, n_total, seed):
    rng = random.Random(seed)
    groups = {
        "vp": [x for x in test_data if x["validity"] and x["plausibility"]],
        "vi": [x for x in test_data if x["validity"] and not x["plausibility"]],
        "ip": [x for x in test_data if not x["validity"] and x["plausibility"]],
        "ii": [x for x in test_data if not x["validity"] and not x["plausibility"]],
    }
    per = max(1, n_total // 4)
    picked = []
    for items in groups.values():
        rng.shuffle(items)
        picked.extend(items[:per])
    rng.shuffle(picked)
    return picked


# ============================================================
# LOAD MODEL (with optional LoRA)
# ============================================================
def load_model(adapter_path=None):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
    )

    if adapter_path is not None:
        from peft import PeftModel
        print(f"Loading LoRA adapter: {adapter_path}")
        model = PeftModel.from_pretrained(model, adapter_path)

    model.eval()
    model.config.use_cache = False
    return model, tokenizer


# ============================================================
# MAIN
# ============================================================
def run_analysis(model, tokenizer, train_data, test_data, tag):
    n_layers = len(get_layers(model))
    out_subdir = os.path.join(OUT_DIR, tag)
    os.makedirs(out_subdir, exist_ok=True)

    validity_examples = stratified_examples(test_data, N_VALIDITY_EXAMPLES, SEED)
    content_pairs = make_content_pairs(test_data, N_CONTENT_PAIRS, SEED + 2)
    mean_examples = stratified_examples(test_data, N_MEAN_EXAMPLES, SEED + 1)

    # --- 1. Validity patching (quantifier-flip corruption) ---
    print(f"\n[{tag}] === VALIDITY PATCHING (quantifier flip) ===")
    val_res, val_logs, val_count = run_validity_patching(
        model, tokenizer, train_data, validity_examples, n_layers
    )
    print(f"Used {val_count}/{len(validity_examples)} examples")
    print(f"Corruption stats: {_corruption_stats}")

    print(f"\n{'Layer':>5} {'Attn':>10} {'MLP':>10}")
    for l in range(n_layers):
        print(f"{l:>5} {val_res['attn'][l]:>10.4f} {val_res['mlp'][l]:>10.4f}")

    save_json({
        "attn": val_res["attn"].tolist(),
        "mlp": val_res["mlp"].tolist(),
        "logs": val_logs,
        "n_used": val_count,
    }, os.path.join(out_subdir, "validity_patching.json"))

    # --- 2. Content-effect patching ---
    print(f"\n[{tag}] === CONTENT-EFFECT PATCHING ===")
    con_res, con_count = run_content_patching(
        model, tokenizer, train_data, content_pairs, n_layers
    )
    print(f"Used {con_count}/{len(content_pairs)} pairs")
    print(f"\n{'Layer':>5} {'Attn':>10} {'MLP':>10}")
    for l in range(n_layers):
        print(f"{l:>5} {con_res['attn'][l]:>10.4f} {con_res['mlp'][l]:>10.4f}")

    save_json({
        "attn": con_res["attn"].tolist(),
        "mlp": con_res["mlp"].tolist(),
        "n_used": con_count,
    }, os.path.join(out_subdir, "content_patching.json"))

    # --- 3. Disentanglement: logic vs content components ---
    # Rank components by each axis; exclude layer 0 (trivial embedding patching)
    logic_scores = []
    content_scores = []
    for l in range(1, n_layers):
        for typ in ["attn", "mlp"]:
            logic_scores.append((l, typ, abs(float(val_res[typ][l]))))
            content_scores.append((l, typ, abs(float(con_res[typ][l]))))
    logic_scores.sort(key=lambda x: x[2], reverse=True)
    content_scores.sort(key=lambda x: x[2], reverse=True)

    print(f"\n[{tag}] Top LOGIC components (high validity-patch, excl layer 0):")
    for l, typ, s in logic_scores[:8]:
        print(f"  L{l:>2} {typ:>4}  {s:.4f}")
    print(f"\n[{tag}] Top CONTENT components (high content-patch, excl layer 0):")
    for l, typ, s in content_scores[:8]:
        print(f"  L{l:>2} {typ:>4}  {s:.4f}")

    save_json({
        "top_logic": [{"layer": l, "type": t, "score": s} for l, t, s in logic_scores[:16]],
        "top_content": [{"layer": l, "type": t, "score": s} for l, t, s in content_scores[:16]],
    }, os.path.join(out_subdir, "top_components.json"))

    top_components = {(l, typ) for l, typ, _ in logic_scores[:8]}

    # --- 4. Mean ablation: sufficiency + necessity ---
    print(f"\n[{tag}] === MEAN ABLATION ===")
    mean_attn, mean_mlp = compute_mean_outputs(
        model, tokenizer, train_data, mean_examples, n_layers
    )
    suff = test_circuit_sufficiency(
        model, tokenizer, train_data, validity_examples[:8],
        top_components, mean_attn, mean_mlp, n_layers,
    )
    print("Sufficiency:", suff)
    nec = test_circuit_necessity(
        model, tokenizer, train_data, validity_examples[:8],
        top_components, mean_attn, mean_mlp,
    )
    print("Necessity:", nec)
    save_json({"sufficiency": suff, "necessity": nec},
              os.path.join(out_subdir, "ablation.json"))

    # --- 5. Head-level content patching on top content-attn layers ---
    print(f"\n[{tag}] === HEAD-LEVEL CONTENT PATCHING ===")
    top_attn_layers = [l for l, t, _ in content_scores if t == "attn"][:N_HEAD_TOP_LAYERS]
    print(f"Top content-attn layers: {top_attn_layers}")
    head_res, head_count, n_heads, head_dim = run_head_level_content_patching(
        model, tokenizer, train_data, content_pairs, top_attn_layers
    )
    print(f"n_heads={n_heads}, head_dim={head_dim}, used {head_count} pairs")
    for l in top_attn_layers:
        row = " ".join(f"{v:+.3f}" for v in head_res[l])
        print(f"  L{l:>2}: {row}")
    save_json(
        {"layers": top_attn_layers,
         "scores": {str(l): head_res[l].tolist() for l in top_attn_layers},
         "n_heads": n_heads, "head_dim": head_dim},
        os.path.join(out_subdir, "head_content_patching.json"),
    )

    # --- 6. Logit lens trajectories ---
    print(f"\n[{tag}] === LOGIT LENS ===")
    traj_results = []
    for ex in validity_examples[:6]:
        prompt = build_prompt(tokenizer, train_data, ex["syllogism"])
        traj = logit_lens_summary(model, tokenizer, prompt, top_k=5)
        traj_results.append({
            "id": ex["id"], "validity": ex["validity"], "plausibility": ex["plausibility"],
            "trajectory": {str(k): v for k, v in traj.items()},
        })
    save_json(traj_results, os.path.join(out_subdir, "logit_lens.json"))

    return {
        "validity": val_res, "content": con_res,
        "head_content": {l: head_res[l].tolist() for l in top_attn_layers},
        "top_attn_layers": top_attn_layers,
    }


def main():
    set_seed(SEED)
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"Device: {DEVICE} | Model: {MODEL_NAME}")

    train_data = load_json(TRAIN_JSON)
    test_data = load_json(TEST_JSON)
    print(f"Train: {len(train_data)} | Test: {len(test_data)}")

    # Base model analysis
    print("\n========== BASE MODEL ==========")
    model, tokenizer = load_model(adapter_path=None)
    yes_id, no_id = yes_no_token_ids(tokenizer)
    print(f"yes={yes_id} {tokenizer.decode([yes_id])!r}  no={no_id} {tokenizer.decode([no_id])!r}")
    base_results = run_analysis(model, tokenizer, train_data, test_data, tag="base")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # LoRA model analysis + comparison
    if LORA_ADAPTER_PATH is not None and os.path.exists(LORA_ADAPTER_PATH):
        print("\n========== LORA MODEL ==========")
        model, tokenizer = load_model(adapter_path=LORA_ADAPTER_PATH)
        lora_results = run_analysis(model, tokenizer, train_data, test_data, tag="lora")

        # Diff: where did plausibility sensitivity drop?
        print("\n========== BASE vs LORA DIFF (content patching) ==========")
        delta_attn = np.abs(base_results["content"]["attn"]) - np.abs(lora_results["content"]["attn"])
        delta_mlp = np.abs(base_results["content"]["mlp"]) - np.abs(lora_results["content"]["mlp"])
        print(f"{'Layer':>5} {'dAttn':>10} {'dMlp':>10}")
        for l in range(len(delta_attn)):
            print(f"{l:>5} {delta_attn[l]:>+10.4f} {delta_mlp[l]:>+10.4f}")
        save_json({
            "delta_attn": delta_attn.tolist(),
            "delta_mlp": delta_mlp.tolist(),
            "interpretation": "positive = LoRA suppressed plausibility sensitivity at this component",
        }, os.path.join(OUT_DIR, "base_vs_lora_content_diff.json"))
    else:
        print("\n(Skipping LoRA comparison: set LORA_ADAPTER_PATH to enable)")

    print(f"\nDone. Results in: {OUT_DIR}")


if __name__ == "__main__":
    main()

Device: cuda | Model: Qwen/Qwen2.5-3B-Instruct
Train: 960 | Test: 191

========== BASE MODEL ==========


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

yes=9834 ' yes'  no=902 ' no'

[base] === VALIDITY PATCHING (quantifier flip) ===
  validity patching 2/12
  validity patching 4/12
  validity patching 8/12
  validity patching 10/12
Used 8/12 examples
Corruption stats: {'success': 8, 'fail': 4}

Layer       Attn        MLP
    0    -0.4809    -0.3847
    1    -0.1796    -0.3884
    2    -0.0217    -0.5395
    3    -0.4308    -0.0162
    4    -0.4099    -0.3725
    5    -0.5416    -0.2264
    6    -0.2541    -0.3932
    7    -0.2626    -0.1598
    8    -0.2406    -0.3945
    9    -0.1784    -0.1615
   10    -0.0607    -0.3928
   11    -0.3700    -0.4988
   12    -0.2880    -0.4569
   13    -0.3975    -0.2932
   14    -0.2626    -0.1783
   15    -0.2613    -0.2918
   16    -0.0746    -0.4265
   17    -0.3894    -0.0436
   18    -0.3339    -0.4873
   19    -0.3235    -0.2039
   20    -0.4522    -0.2697
   21    -0.5494    -0.4158
   22    -0.1899    -0.2577
   23    -0.1322    -0.1336
   24     0.0888     0.4453
   25    -0.4082    -0.10